### Understanding Tokenization in Transformers

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')

In [3]:
import math
import sys, os

In [4]:
import torch
import torch.nn as nn
from torchtext.datasets import Multi30k

### Step-0:- Load the Dataset
The dataset is used for language translation purposes

In [5]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# ! wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [6]:

from datasets import load_dataset
from pathlib import Path

DATASET_NAME = "opus_books" 

In [7]:
ds = load_dataset(DATASET_NAME, "en-it", split="train")
len(ds), type(ds), ds[3]

(32332,
 datasets.arrow_dataset.Dataset,
 {'id': '3', 'translation': {'en': 'CHAPTER I', 'it': 'PARTE PRIMA'}})

In [8]:
import random
examples = random.sample(range(len(ds)), 3)
src_text = ds[examples[0]]["translation"]["en"]
tgt_text = ds[examples[0]]["translation"]["it"]
print("One Example:")
print("English:", src_text)
print("Italian:", tgt_text)
print("-----------")
for i in examples:
    print(ds[i])
    print("---")

One Example:
English: Though the first family was all right, Prince Chechensky felt happier with the second family.
Italian: Sebbene anche la prima famiglia fosse buona, il principe cecenskij si sentiva più felice nella seconda famiglia.
-----------
{'id': '30366', 'translation': {'en': 'Though the first family was all right, Prince Chechensky felt happier with the second family.', 'it': 'Sebbene anche la prima famiglia fosse buona, il principe cecenskij si sentiva più felice nella seconda famiglia.'}}
---
{'id': '19647', 'translation': {'en': "'Allow me to introduce myself,' said the Princess with her dignified smile. 'My daughter has fallen in love with you.", 'it': '— Permettetemi di fare la vostra conoscenza — disse con il suo sorriso sostenuto. — Mia figlia è innamorata di voi.'}}
---
{'id': '18516', 'translation': {'en': 'While they were talking Laska, pricking her ears, kept looking up at the sky and then reproachfully at them.', 'it': 'Mentre dicevano questo, Laska, drizzando l

### Step-1:- Load Dataset and Tokenize it

In [9]:
from src.config import get_config
from src.dataset_path import DATASET_DIR, get_or_build_tokenizer, get_all_sentences, BilingualDataset

In [10]:
sentences = get_all_sentences(ds, "en")
print(next(sentences))
print(next(sentences))
print(next(sentences))

Source: Project Gutenberg
Jane Eyre
Charlotte Bronte


In [11]:
config = get_config()
config

{'batch_size': 8,
 'num_epochs': 20,
 'lr': 0.0001,
 'seq_len': 350,
 'd_model': 512,
 'datasource': 'opus_books',
 'lang_src': 'en',
 'lang_tgt': 'it',
 'model_folder': 'weights',
 'model_basename': 'tmodel_',
 'preload': 'latest',
 'tokenizer_file': 'tokenizer_{0}.json',
 'experiment_name': 'runs/tmodel'}

In [12]:
tokenizer_src = get_or_build_tokenizer(
    config,
    ds,
    config["lang_src"]
)

tokenizer_tgt = get_or_build_tokenizer(
    config,
    ds,
    config["lang_tgt"]
)

In [13]:
print(tokenizer_src.get_vocab_size())
for k, v in list(tokenizer_src.get_vocab().items())[:10]:
    print(k, v)

15698
pencils 6645
competency 10595
grove 4548
Bending 8635
earthen 4260
lusty 11217
misapprehension 14400
stimulated 8457
ploughshares 14671
barber 7800


In [14]:
for token in ["[UNK]", "[PAD]", "[SOS]", "[EOS]"]:
    print(token, tokenizer_src.token_to_id(token))

[UNK] 0
[PAD] 3
[SOS] 1
[EOS] 2


In [15]:
SAMPLE =10
SAMPLE_DATA = ds[SAMPLE]

In [16]:
sentence = SAMPLE_DATA["translation"]["en"]

encoded = tokenizer_src.encode(sentence)

print("Sentence:", sentence)
print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)

Sentence: "Jane, I don't like cavillers or questioners; besides, there is something truly forbidding in a child taking up her elders in that manner.
Tokens: ['"', 'Jane', ',', 'I', 'don', "'", 't', 'like', '[UNK]', 'or', '[UNK]', ';', 'besides', ',', 'there', 'is', 'something', 'truly', 'forbidding', 'in', 'a', 'child', 'taking', 'up', 'her', 'elders', 'in', 'that', 'manner', '.']
IDs: [35, 298, 4, 9, 169, 12, 83, 96, 0, 54, 0, 18, 746, 4, 64, 34, 183, 2332, 13816, 13, 11, 411, 442, 56, 23, 10815, 13, 15, 493, 7]


In [17]:
src_text = SAMPLE_DATA["translation"]["en"]
tgt_text = SAMPLE_DATA["translation"]["it"]

src_ids = tokenizer_src.encode(src_text).ids
tgt_ids = tokenizer_tgt.encode(tgt_text).ids

print(src_text)
print(src_ids)

print(tgt_text)
print(tgt_ids)

"Jane, I don't like cavillers or questioners; besides, there is something truly forbidding in a child taking up her elders in that manner.
[35, 298, 4, 9, 169, 12, 83, 96, 0, 54, 0, 18, 746, 4, 64, 34, 183, 2332, 13816, 13, 11, 411, 442, 56, 23, 10815, 13, 15, 493, 7]
— Jane, non mi piace di essere interrogata. Sta male, del resto, che una bimba tratti così i suoi superiori.
[9, 269, 4, 12, 25, 791, 7, 90, 11799, 5, 4348, 306, 4, 31, 535, 4, 8, 21, 1471, 1251, 55, 32, 157, 4565, 5]


In [18]:
seq_len = 20

src_sos = tokenizer_src.token_to_id("[SOS]")
src_eos = tokenizer_src.token_to_id("[EOS]")
src_pad = tokenizer_src.token_to_id("[PAD]")
print(src_sos, src_eos, src_pad)

tgt_sos = tokenizer_tgt.token_to_id("[SOS]")
tgt_eos = tokenizer_tgt.token_to_id("[EOS]")
tgt_pad = tokenizer_tgt.token_to_id("[PAD]")
print(tgt_sos, tgt_eos, tgt_pad)

1 2 3
1 2 3


In [19]:
encoder_input = torch.tensor(
    [src_sos]
    + src_ids
    + [src_eos]
    + [src_pad] * (seq_len - len(src_ids) - 2)
)
encoder_input

tensor([    1,    35,   298,     4,     9,   169,    12,    83,    96,     0,
           54,     0,    18,   746,     4,    64,    34,   183,  2332, 13816,
           13,    11,   411,   442,    56,    23, 10815,    13,    15,   493,
            7,     2])

In [20]:

decoder_input = torch.tensor(
    [tgt_sos]
    + tgt_ids
    + [tgt_pad] * (seq_len - len(tgt_ids) - 1)
)
decoder_input

tensor([    1,     9,   269,     4,    12,    25,   791,     7,    90, 11799,
            5,  4348,   306,     4,    31,   535,     4,     8,    21,  1471,
         1251,    55,    32,   157,  4565,     5])

In [21]:


label = torch.tensor(
    tgt_ids
    + [tgt_eos]
    + [tgt_pad] * (seq_len - len(tgt_ids) - 1)
)
label

tensor([    9,   269,     4,    12,    25,   791,     7,    90, 11799,     5,
         4348,   306,     4,    31,   535,     4,     8,    21,  1471,  1251,
           55,    32,   157,  4565,     5,     2])

In [22]:
print("encoder:", encoder_input)
print("decoder:", decoder_input)
print("label:  ", label)

print(encoder_input.shape)
print(decoder_input.shape)
print(label.shape)

encoder: tensor([    1,    35,   298,     4,     9,   169,    12,    83,    96,     0,
           54,     0,    18,   746,     4,    64,    34,   183,  2332, 13816,
           13,    11,   411,   442,    56,    23, 10815,    13,    15,   493,
            7,     2])
decoder: tensor([    1,     9,   269,     4,    12,    25,   791,     7,    90, 11799,
            5,  4348,   306,     4,    31,   535,     4,     8,    21,  1471,
         1251,    55,    32,   157,  4565,     5])
label:   tensor([    9,   269,     4,    12,    25,   791,     7,    90, 11799,     5,
         4348,   306,     4,    31,   535,     4,     8,    21,  1471,  1251,
           55,    32,   157,  4565,     5,     2])
torch.Size([32])
torch.Size([26])
torch.Size([26])


In [23]:
print(tokenizer_src.decode(encoder_input.tolist()))
print(tokenizer_tgt.decode(decoder_input.tolist()))
print(tokenizer_tgt.decode(label.tolist()))

" Jane , I don ' t like or ; besides , there is something truly forbidding in a child taking up her elders in that manner .
— Jane , non mi piace di essere interrogata . Sta male , del resto , che una bimba tratti così i suoi superiori .
— Jane , non mi piace di essere interrogata . Sta male , del resto , che una bimba tratti così i suoi superiori .


In [24]:
train_ds = BilingualDataset(
    ds,
    tokenizer_src,
    tokenizer_tgt,
    "en",
    "it",
    seq_len=350
)

In [25]:
sample = train_ds[10]

for key, value in sample.items():
    if torch.is_tensor(value):
        print(key, value.shape)
    else:
        print(key, value)

encoder_input torch.Size([350])
decoder_input torch.Size([350])
encoder_mask torch.Size([1, 1, 350])
decoder_mask torch.Size([1, 350, 350])
label torch.Size([350])
src_text "Jane, I don't like cavillers or questioners; besides, there is something truly forbidding in a child taking up her elders in that manner.
tgt_text — Jane, non mi piace di essere interrogata. Sta male, del resto, che una bimba tratti così i suoi superiori.


In [26]:
print(tokenizer_src.decode(sample["encoder_input"].tolist()))
print(tokenizer_tgt.decode(sample["decoder_input"].tolist()))
print(tokenizer_tgt.decode(sample["label"].tolist()))

" Jane , I don ' t like or ; besides , there is something truly forbidding in a child taking up her elders in that manner .
— Jane , non mi piace di essere interrogata . Sta male , del resto , che una bimba tratti così i suoi superiori .
— Jane , non mi piace di essere interrogata . Sta male , del resto , che una bimba tratti così i suoi superiori .


In [27]:
print(
    tokenizer_tgt.decode(
        sample["decoder_input"].tolist(),
        skip_special_tokens=False
    )
)

print(
    tokenizer_tgt.decode(
        sample["label"].tolist(),
        skip_special_tokens=False
    )
)

[SOS] — Jane , non mi piace di essere interrogata . Sta male , del resto , che una bimba tratti così i suoi superiori . [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD

In [28]:
print("decoder:", sample["decoder_input"][:10])
print("label:  ", sample["label"][:10])

print("decoder end:", sample["decoder_input"][-10:])
print("label end:  ", sample["label"][-10:])

decoder: tensor([    1,     9,   269,     4,    12,    25,   791,     7,    90, 11799])
label:   tensor([    9,   269,     4,    12,    25,   791,     7,    90, 11799,     5])
decoder end: tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3])
label end:   tensor([3, 3, 3, 3, 3, 3, 3, 3, 3, 3])


#### Embeddings 

In [34]:
import torch
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=10,
    embedding_dim=4
)

print(embedding.weight.shape)
print(embedding.weight)

torch.Size([10, 4])
Parameter containing:
tensor([[ 0.3726,  0.9258, -0.1808, -0.2866],
        [-1.1041, -0.1924,  1.8335, -0.2754],
        [-0.0399,  1.5948, -0.4564, -1.3120],
        [-0.3738,  1.3831,  0.1373, -0.9510],
        [-0.6615, -0.8415,  0.5318, -0.5374],
        [-0.4519, -0.3372,  1.2002, -2.1612],
        [ 1.4346, -0.3765,  0.1461, -0.2984],
        [-1.5821, -0.4772,  0.1210,  0.0835],
        [-0.3032,  0.0029,  0.8308,  1.7503],
        [-1.2788,  0.5139,  2.3332, -0.2992]], requires_grad=True)


In [35]:
x = torch.tensor([2, 5, 2])

out = embedding(x)

print(x.shape)
print(out.shape)
print(out)

torch.Size([3])
torch.Size([3, 4])
tensor([[-0.0399,  1.5948, -0.4564, -1.3120],
        [-0.4519, -0.3372,  1.2002, -2.1612],
        [-0.0399,  1.5948, -0.4564, -1.3120]], grad_fn=<EmbeddingBackward0>)


In [36]:
print(out[0])
print(out[2])

assert torch.equal(out[0], out[2])

tensor([-0.0399,  1.5948, -0.4564, -1.3120], grad_fn=<SelectBackward0>)
tensor([-0.0399,  1.5948, -0.4564, -1.3120], grad_fn=<SelectBackward0>)


In [29]:
### Our requirement is Dimension size and vocab size
DIMENSION_SIZE = 512

In [37]:
from src.dataset_path import DATASET_DIR, complete_tokenization

In [38]:
train_dataloader,val_dataloader, tokenizer_src, tokenizer_tgt = complete_tokenization(config, ds_raw=None)

Max Length of Source Sentence: 309
Max Length of Target Sentence: 274


In [39]:
sample_batch = next(iter(val_dataloader))
sample_batch.keys()

dict_keys(['encoder_input', 'decoder_input', 'encoder_mask', 'decoder_mask', 'label', 'src_text', 'tgt_text'])

In [30]:
from src.input_embd import InputEmbedding

In [43]:
embd_layer = InputEmbedding(dimension_size=DIMENSION_SIZE, vocab_size=tokenizer_src.get_vocab_size())
embd_layer

InputEmbedding(
  (embedding_layer): Embedding(15698, 512)
)

In [44]:
print(sample_batch['encoder_input'].shape)
embd = embd_layer(sample_batch['encoder_input'])
print(embd.shape)

torch.Size([1, 350])
torch.Size([1, 350, 512])


In [ ]:
embd[0]

tensor([[[ 10.8685,  13.6885,  20.4335,  ..., -28.1426, -15.2918,   2.9192],
         [-46.0382,  -8.6546,  10.7229,  ...,  25.5041, -57.4352,  -1.7143],
         [-28.7495,   1.3557, -19.7802,  ..., -24.6348,  -9.1902,  -4.0068],
         ...,
         [-23.0269,  -9.9804,  57.3971,  ..., -42.7777, -25.8419,  17.4739],
         [-23.0269,  -9.9804,  57.3971,  ..., -42.7777, -25.8419,  17.4739],
         [-23.0269,  -9.9804,  57.3971,  ..., -42.7777, -25.8419,  17.4739]]],
       grad_fn=<MulBackward0>)

In [42]:
print("Token ID:", sample_batch['encoder_input'][0, 1])
print("Token:", tokenizer_src.id_to_token(sample_batch['encoder_input'][0, 1].item()))

print("Embedding:")
print(embd[0, 0])

print("Embedding shape:", embd[0, 0].shape)

Token ID: tensor(62)
Token: The
Embedding:
tensor([ 3.0026e+01, -1.4850e+01, -1.1417e+01, -1.6559e+01, -8.7156e+00,
        -2.8744e+01, -7.8314e+00,  2.9094e+01,  4.5450e+01, -2.3342e-01,
        -2.0185e+01,  3.7605e+01, -2.9118e+01,  1.0415e+00, -1.5313e+01,
        -1.7623e+01, -1.9063e+01, -1.4758e+01,  8.3850e+00,  2.1113e+01,
         4.2333e+00,  1.1718e+00, -3.4071e+01,  3.0516e+00,  2.4194e+01,
         3.6343e+01, -3.6036e+01, -1.3823e+00, -3.6459e+01, -2.7099e+00,
         9.3536e+00, -1.7719e+01,  2.5162e+01, -8.3054e+00,  3.1218e+00,
        -2.3476e+01, -4.7623e+00,  3.4647e+01,  7.7745e+00, -4.5892e+01,
         9.0904e+00,  1.0231e+01, -1.4010e+01, -3.0766e+01,  1.7001e+01,
        -5.4403e+00, -1.8570e+01,  1.6017e+01, -1.0194e+01, -6.3572e-02,
        -2.7456e+01,  1.6370e+01, -1.2536e+01, -1.2763e+01,  2.5233e+01,
        -3.9701e+00, -2.7773e+01, -5.1635e+01, -3.8004e+01, -2.1741e+01,
        -2.7427e+01, -1.9979e+01,  1.6480e+00,  2.0193e+01, -4.0205e+01,
        